# EDA tầng Bronze — OutbreakSignal DE

**Chạy trên Google Colab, không cần cài gì trên máy.** Notebook tự tải dữ liệu từ
đúng các nguồn mà pipeline dùng, dựng tầng Bronze ngay trong Colab, rồi phân tích
**cả 3 nguồn**: OpenDengue, Google News RSS, WHO GHO.

Mục tiêu: trả lời bốn câu hỏi trước khi thiết kế tầng Silver.

1. Mỗi bảng có **những cột gì**, kiểu gì, null bao nhiêu, trùng lặp bao nhiêu?
2. Dữ liệu phủ tới **đơn vị địa lý nào** — quốc gia, tỉnh, hay nhỏ hơn?
3. **Giá trị** trông ra sao — thiếu, âm, trùng lặp, lệch phân bố?
4. Ba nguồn có **ghép được với nhau** không?

> ⚠️ EDA chỉ **đọc** Bronze, không ghi đè hay sửa gì. Nguyên tắc "Bronze giữ
> nguyên trạng nguồn" vẫn được tôn trọng — không phép biến đổi nào được ghi
> ngược trở lại.

Thời gian chạy: khoảng 6–9 phút (phần lớn là cài PySpark và tải OpenDengue Spatial
extract, ~55MB).


---
## Phần 0 — Cài đặt

Bản `pyspark` phải là **3.5.9**: các bản 3.5.x cũ hơn dính
[SPARK-53759](https://issues.apache.org/jira/browse/SPARK-53759).


In [ ]:
!pip install -q pyspark==3.5.9 delta-spark==3.3.3
print('Xong. Chạy tiếp cell sau.')


In [ ]:
import json
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import requests
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Bảng màu đã kiểm định (tương phản + mù màu). Ba slot đầu dùng được cho mọi
# loại biểu đồ; đừng tự đổi sang màu khác.
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED = '#0b0b0b', '#52514e'

def style_axes(ax, title, xlabel=''):
    """Lưới mờ, bỏ khung, chữ dùng màu chữ chứ không dùng màu series."""
    ax.set_title(title, color=INK, fontsize=13, pad=14, loc='left')
    ax.set_xlabel(xlabel, color=MUTED, fontsize=10)
    ax.tick_params(colors=MUTED, labelsize=10, length=0)
    for side in ('top', 'right', 'left', 'bottom'):
        ax.spines[side].set_visible(False)
    ax.grid(axis='x', color='#e5e5e2', linewidth=0.8)
    ax.set_axisbelow(True)

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
print('Import xong.')


In [ ]:
builder = (
    SparkSession.builder.appName('eda_bronze_colab')
    .master('local[*]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.ui.showConsoleProgress', 'false')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)


---
## Phần 1 — Dựng Bronze trong Colab

Cùng URL, cùng quy tắc với pipeline thật (`configs/sources.yaml`):
`inferSchema=False`, không đổi tên cột, không lọc, không ép kiểu. Chỉ thêm cột
truy vết.

Nếu đã có sẵn `data/bronze/` từ máy, xem **Phụ lục** ở cuối để upload và bỏ qua
phần này.


In [ ]:
SOURCES = {
    'opendengue': {
        # Spatial_extract, KHONG phai National_extract - can cap tinh de tra
        # loi cau hoi 'vung nao', dung URL that pipeline dang dung.
        'url': ('https://github.com/OpenDengue/master-repo/raw/main/data/'
                'releases/V1.3/Spatial_extract_V1_3.zip'),
    },
    'news_rss': {
        'url': 'https://news.google.com/rss/search',
        'params': {'q': 'dengue Southeast Asia', 'hl': 'en',
                   'gl': 'US', 'ceid': 'US:en'},
    },
    'who_gho': {
        'url': ('https://xmart-api-public.who.int/ARBOV/'
                'V_DENGUE_GLOBAL_VALIDATED_PUBLIC'),
    },
}

# 11 nuoc Dong Nam A - dung de loc pham vi thu thap, giong
# configs/sources.yaml cua pipeline that. OpenDengue loc theo ten
# (adm_0_name), WHO GHO loc theo ma ISO3.
SEA_COUNTRIES = ['VIET NAM', 'THAILAND', 'INDONESIA', 'PHILIPPINES', 'MALAYSIA',
                 'MYANMAR', 'CAMBODIA', "LAO PEOPLE'S DEMOCRATIC REPUBLIC",
                 'SINGAPORE', 'BRUNEI DARUSSALAM', 'TIMOR-LESTE']
SEA_ISO3 = ['VNM', 'THA', 'IDN', 'PHL', 'MYS', 'MMR', 'KHM', 'LAO', 'SGP', 'BRN', 'TLS']

LANDING = Path('/content/data/landing')
BRONZE = Path('/content/data/bronze')
TODAY = datetime.now(timezone.utc).strftime('%Y-%m-%d')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def landing_dir(source):
    path = LANDING / source / TODAY
    path.mkdir(parents=True, exist_ok=True)
    return path

def write_bronze(df, source):
    """Thêm cột truy vết rồi ghi Delta. Không đụng vào cột gốc."""
    enriched = (df
        .withColumn('_source', F.lit(source))
        .withColumn('_ingested_at', F.lit(datetime.now(timezone.utc).isoformat()))
        .withColumn('ingestion_date', F.lit(TODAY)))
    (enriched.write.format('delta').mode('overwrite')
        .partitionBy('ingestion_date').save(str(BRONZE / source)))
    return enriched

print('Cấu hình xong. ingestion_date =', TODAY)


In [ ]:
# --- OpenDengue: zip -> CSV -> loc SEA -> Bronze ---
# File Spatial_extract ~55MB, tai lau hon National truoc day (~5-7 phut).
dest = landing_dir('opendengue')
resp = requests.get(SOURCES['opendengue']['url'], timeout=180)
resp.raise_for_status()
zip_path = dest / 'Spatial_extract_V1_3.zip'
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path) as archive:
    csv_name = [n for n in archive.namelist() if n.lower().endswith('.csv')][0]
    archive.extract(csv_name, dest)

# inferSchema=False -> tất cả là string. CỐ Ý: Bronze không ép kiểu.
od_raw = spark.read.csv(str(dest / '*.csv'), header=True, inferSchema=False)
print(f'Truoc khi loc: {od_raw.count():,} dòng toàn cầu')

# Loc theo pham vi SEA TRUOC KHI ghi Bronze - giong dung ingestion/opendengue.py
# cua pipeline that. Day la quyet dinh PHAM VI THU THAP, khong phai transformation:
# khong doi gia tri dong nao con lai, chi chon dong nao duoc dua vao kho. Ly do bat
# buoc: file goc 2,8 trieu dong toan cau, SEA chi chiem ~2,5%.
od = write_bronze(od_raw.filter(F.col('adm_0_name').isin(SEA_COUNTRIES)), 'opendengue')
print(f'opendengue (đã lọc SEA): {od.count():,} dòng, {len(od.columns)} cột')


In [ ]:
# --- WHO GHO: OData filter server-side -> JSONL -> Bronze ---
cfg = SOURCES['who_gho']
dest = landing_dir('who_gho')
quoted = ','.join(f"'{iso}'" for iso in SEA_ISO3)
params = {'$filter': f'ISO3 in ({quoted})', '$top': 10_000}
resp = requests.get(cfg['url'], params=params, timeout=30)
resp.raise_for_status()
(dest / f'who_gho_{TODAY}.json').write_bytes(resp.content)

records = resp.json().get('value', [])
with open(dest / f'who_gho_{TODAY}.jsonl', 'w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

who = write_bronze(spark.read.json(str(dest / '*.jsonl')), 'who_gho')
print(f'who_gho (đã lọc SEA server-side): {who.count():,} dòng, {len(who.columns)} cột')


In [ ]:
# --- Google News RSS: XML -> JSONL -> Bronze ---
cfg = SOURCES['news_rss']
dest = landing_dir('news_rss')
resp = requests.get(cfg['url'], params=cfg['params'],
                    headers={'User-Agent': 'OutbreakSignalDE/0.1'}, timeout=60)
resp.raise_for_status()
(dest / f'google_news_{STAMP}.xml').write_bytes(resp.content)

FIELDS = ('title', 'link', 'pubDate', 'source', 'description')
items = [{f: (it.findtext(f) or '').strip() for f in FIELDS}
         for it in ET.fromstring(resp.content).findall('.//item')]

with open(dest / f'google_news_{STAMP}.jsonl', 'w', encoding='utf-8') as handle:
    for item in items:
        handle.write(json.dumps(item, ensure_ascii=False) + '\n')

news = write_bronze(spark.read.json(str(dest / '*.jsonl')), 'news_rss')
print(f'news_rss: {news.count():,} dòng, {len(news.columns)} cột')


---
## Phần 1.5 — Profiling tổng quát: cột, null, trùng lặp

Trước khi đi vào từng câu hỏi cụ thể, quét một lượt CHUNG cho cả ba bảng:
mỗi bảng có cột gì, kiểu gì, null bao nhiêu, có bao nhiêu dòng trùng lặp hoàn
toàn. Đây là bước nên làm ĐẦU TIÊN khi nhận một bảng mới — nó cho biết nên hỏi
tiếp câu gì, thay vì đoán trước rồi mới đi tìm bằng chứng.


In [ ]:
def profile_columns(df, name):
    """In: tên cột, kiểu, số null, %null, số giá trị phân biệt."""
    total = df.count()
    print()
    print('=' * 70)
    print(f'PROFILE: {name}  ({total:,} dòng, {len(df.columns)} cột)')
    print('=' * 70)
    rows = []
    for field in df.schema.fields:
        col = field.name
        nulls = df.filter(F.col(col).isNull()).count()
        distinct = df.select(col).distinct().count()
        rows.append((col, field.dataType.simpleString(), nulls,
                     round(100 * nulls / total, 1) if total else 0.0, distinct))
    width = max(len(r[0]) for r in rows) + 2
    header = ('cot'.ljust(width) + 'kieu'.ljust(12) + 'null'.rjust(8)
              + '%null'.rjust(8) + 'distinct'.rjust(10))
    print(header)
    for col, dtype, nulls, pct, distinct in rows:
        line = (col.ljust(width) + dtype.ljust(12) + str(nulls).rjust(8)
                + f'{pct}%'.rjust(8) + str(distinct).rjust(10))
        print(line)

def check_duplicates(df, name, key_cols=None):
    """So dong trung lap hoan toan, va trung theo key_cols neu co truyen vao."""
    total = df.count()
    distinct_rows = df.distinct().count()
    print()
    print(f'--- Trùng lặp trong {name} ---')
    print(f'  Dòng trùng TOÀN BỘ cột: {total - distinct_rows:,} / {total:,}')
    if key_cols:
        key_distinct = df.select(*key_cols).distinct().count()
        print(f'  Trùng theo khoá {key_cols}: {total - key_distinct:,} / {total:,}')

for frame, name, key in ((od, 'opendengue', None),
                          (news, 'news_rss', ['link']),
                          (who, 'who_gho', ['ISO3', 'START_DATE', 'DATE_TYPE'])):
    profile_columns(frame, name)
    check_duplicates(frame, name, key_cols=key)


**Đọc bảng profile như thế nào:**

- Cột `%null` cao đột biến (như `POPULATION` của `who_gho`) là dấu hiệu cột đó
  gần như không có dữ liệu thật — cần soi kỹ ở phần sau, đừng vội dùng.
- Cột `distinct` = 1 nghĩa là cột hằng số trên toàn bảng — vô dụng để lọc/nhóm.
- `news_rss` được kỳ vọng có dòng trùng theo `link` (mỗi lần fetch là một quan
  sát riêng, đúng thiết kế Bronze) — con số ở đây định lượng việc Silver phải
  dedup, không phải một lỗi.
- `opendengue`, `who_gho` không nên có trùng lặp toàn bộ cột — nếu có,
  đó là dấu hiệu bug ở bước ghi Bronze (vd. `replaceWhere` không lọc đúng phân vùng,
  hoặc đặt tên file landing theo run_id thay vì theo ngày cho nguồn dạng snapshot —
  đây chính xác là bug thật đã gặp và sửa ở `who_gho`).


---
## Phần 2 — OpenDengue

`od` ở đây **đã được lọc SEA** ngay từ bước fetch (Phần 1) — giống hệt pipeline
thật, không phải toàn bộ 2,8 triệu dòng toàn cầu.

### 2.1 Dữ liệu phủ tới đơn vị địa lý nào?

Bảng có ba cột `adm_0_name` / `adm_1_name` / `adm_2_name` (quốc gia / tỉnh /
huyện). Dùng `Spatial_extract` (không phải `National_extract`) nên **có thật** dữ
liệu dưới cấp quốc gia — khác với phát hiện ban đầu khi nhóm còn dùng
`National_extract`.


In [ ]:
print('--- Số giá trị phân biệt của từng cấp hành chính ---')
od.select(
    F.countDistinct('adm_0_name').alias('so_quoc_gia'),
    F.countDistinct('adm_1_name').alias('so_tinh'),
    F.countDistinct('adm_2_name').alias('so_huyen'),
).show()

print('--- S_res: độ phân giải không gian nguồn tự khai báo ---')
od.groupBy('S_res').count().orderBy(F.desc('count')).show(truncate=False)


**Kết luận 2.1 — có dữ liệu cấp tỉnh thật, không phải chỉ cấp quốc gia.**

Đủ 11/11 nước SEA, và có hàng trăm tỉnh phân biệt (`adm_1_name`), hàng chục huyện
(`adm_2_name`). Phần lớn số dòng là `Admin1` (cấp tỉnh) — chỉ một phần nhỏ là
`Admin0` (cấp quốc gia, dùng để tổng hợp nhanh hoặc khi nguồn không có chi tiết
hơn cho giai đoạn đó).

> **Đối chiếu với phát hiện cũ:** `National_extract` (bản trước đây) có
> `adm_1_name`/`adm_2_name` toàn chuỗi `"NA"` — không dùng được cho "khoanh vùng
> nguy cơ". Đây là lý do pipeline chuyển sang `Spatial_extract`, xem
> `README.md` mục 4.1.


### 2.2 `T_res` — độ phân giải thời gian, và một phát hiện tinh vi hơn dự kiến

Cột `T_res` cho biết mỗi dòng là số liệu của một **tuần**, một **tháng**, hay cả
một **năm**. Câu hỏi: chúng có **chồng lên nhau** trong cùng một (nước, tỉnh,
huyện, năm) không? Nếu có thì `SUM` sẽ đếm trùng.

> Lưu ý: ở bản `National_extract` cũ (chỉ cấp quốc gia), nhóm từng kết luận **không
> bao giờ** chồng lấn. Kiểm tra lại với dữ liệu có cấp tỉnh cho kết quả **khác**.


In [ ]:
od.groupBy('T_res').count().orderBy(F.desc('count')).show(truncate=False)

print('--- Mỗi tổ hợp (nước, tỉnh, huyện, năm) có mấy loại T_res? ---')
combo = (od.groupBy('adm_0_name', 'adm_1_name', 'adm_2_name', 'Year')
           .agg(F.countDistinct('T_res').alias('so_loai_T_res')))
(combo.groupBy('so_loai_T_res').agg(F.count('*').alias('so_to_hop'))
      .orderBy('so_loai_T_res').show(truncate=False))


**Kết luận 2.2 — CÓ chồng lấn, ở một phần nhỏ nhưng không phải bằng 0.**

Khác với kết luận cũ (đúng cho `National_extract`, chỉ có cấp quốc gia): khi tính
cả cấp tỉnh/huyện, một phần các tổ hợp (nước, tỉnh, huyện, năm) có **2 loại `T_res`
cùng lúc**. `SUM(dengue_total)` không lọc `T_res` trước sẽ đếm trùng ở đúng những
tổ hợp này — không phải toàn bộ bảng, nhưng đủ để sai nếu bỏ qua.

Bài học: một kết luận đúng ở độ phân giải không gian này **không tự động đúng** ở
độ phân giải khác — phải kiểm tra lại, không suy diễn.


In [ ]:
print('--- Ví dụ cụ thể: một tỉnh Việt Nam, T_res theo từng năm ---')
mau_tinh = (od.filter((F.col('adm_0_name') == 'VIET NAM') &
                      (F.col('S_res') == 'Admin1'))
              .select('adm_1_name').distinct().first()['adm_1_name'])
print(f'Tỉnh mẫu: {mau_tinh}')
(od.filter((F.col('adm_0_name') == 'VIET NAM') & (F.col('adm_1_name') == mau_tinh))
   .groupBy('Year').pivot('T_res', ['Week', 'Month', 'Year']).count()
   .orderBy('Year').show(40, truncate=False))

print('--- Việt Nam cấp TỈNH nói chung: T_res nào chiếm đa số? ---')
(od.filter((F.col('adm_0_name') == 'VIET NAM') & (F.col('S_res') == 'Admin1'))
   .groupBy('T_res').count().orderBy(F.desc('count')).show())


> **Ảnh hưởng tới Silver:** cấp tỉnh của Việt Nam trong mẫu trên chủ yếu ở độ phân
> giải **tháng**, khác với cấp quốc gia (đã có tuần từ 2016). Không thể giả định
> "cấp tỉnh cũng mịn như cấp quốc gia" — phải kiểm tra riêng cho từng cấp trước
> khi thiết kế grain thời gian thống nhất ở Silver.


### 2.3 Phủ Đông Nam Á — bug đặt tên đã sửa ở bước fetch

Tên nước trong dataset theo chuẩn UN, **không** phải cách viết thông thường. Vì
`od` đã lọc bằng `SEA_COUNTRIES` (dùng đúng `"VIET NAM"` có dấu cách) ngay từ
Phần 1, bug này **không còn ảnh hưởng** tới bảng hiện tại — nhưng vẫn đáng ghi lại
vì đã từng gây mất dữ liệu âm thầm.


In [ ]:
print('--- Nếu lọc bằng "VIETNAM" viết liền (cách spikes/ TỪNG dùng) ---')
print('số dòng nếu dùng bug cũ:', od.filter(F.col('adm_0_name') == 'VIETNAM').count())

print()
print('--- Nguồn thực sự viết thế nào? ---')
od.select('adm_0_name').distinct().filter(F.col('adm_0_name').rlike('(?i)viet')).show(truncate=False)

print('--- od đã lọc đúng, Việt Nam CÓ mặt: ---')
print('số dòng Việt Nam trong od:', od.filter(F.col('adm_0_name') == 'VIET NAM').count())


> 🐛 **Lịch sử lỗi (đã sửa).** `spikes/test_opendengue.py` từng lọc bằng
> `"VIETNAM"` viết liền, trong khi nguồn ghi `"VIET NAM"` có dấu cách. Việt Nam bị
> loại âm thầm — đó là lý do ghi chú tiến độ tuần trước nói "10 nước" thay vì 11.
> `configs/sources.yaml` và spike script đã sửa dùng `"VIET NAM"` đúng.


In [ ]:
(od.groupBy('adm_0_name')
   .agg(F.count('*').alias('so_dong'),
        F.min('calendar_start_date').alias('tu_ngay'),
        F.max('calendar_end_date').alias('den_ngay'))
   .orderBy(F.desc('so_dong')).show(20, truncate=False))


#### Biểu đồ — số dòng theo nước, tách theo độ phân giải thời gian

Cố ý vẽ **số dòng** chứ không vẽ tổng số ca: tổng số ca có thể bị đếm trùng ở một
phần tổ hợp, theo phát hiện ở mục 2.2.


In [ ]:
pivot = (od.groupBy('adm_0_name').pivot('T_res', ['Week', 'Month', 'Year'])
            .count().fillna(0).toPandas().set_index('adm_0_name'))
pivot['tong'] = pivot.sum(axis=1)
pivot = pivot.sort_values('tong')

fig, ax = plt.subplots(figsize=(9, 5.5))
left = [0] * len(pivot)
for col, color in zip(['Week', 'Month', 'Year'], [BLUE, ORANGE, AQUA]):
    ax.barh(pivot.index, pivot[col], left=left, height=0.62, color=color,
            label=col, edgecolor='white', linewidth=2)
    left = [a + b for a, b in zip(left, pivot[col])]

# Nhãn giá trị hiện rõ: bắt buộc, vì có màu tương phản thấp trên nền sáng.
for y, total in enumerate(pivot['tong']):
    ax.text(total + 8, y, f'{int(total)}', va='center', color=MUTED, fontsize=9)

style_axes(ax, 'Số dòng OpenDengue theo nước, tách theo độ phân giải thời gian',
           'số dòng')
ax.legend(frameon=False, loc='lower right', fontsize=10, labelcolor=MUTED)
ax.set_xlim(0, pivot['tong'].max() * 1.12)
plt.tight_layout()
plt.show()


### 2.4 Chất lượng giá trị `dengue_total`


In [ ]:
od.select(
    F.count('*').alias('tong_dong'),
    F.count(F.when(F.col('dengue_total').isNull(), 1)).alias('null'),
    F.count(F.when(F.trim('dengue_total') == '', 1)).alias('chuoi_rong'),
    F.count(F.when(F.col('dengue_total').cast('double') < 0, 1)).alias('gia_tri_am'),
    F.count(F.when(F.col('dengue_total').isNotNull() &
                   F.col('dengue_total').cast('double').isNull(), 1)
            ).alias('khong_ep_kieu_duoc'),
).show()


---
## Phần 3 — Google News RSS

### 3.1 Trùng lặp giữa các lần fetch


In [ ]:
print('tổng số dòng    :', news.count())
print('số link duy nhất:', news.select('link').distinct().count())

print()
print('--- một bài xuất hiện mấy lần? ---')
(news.groupBy('link').count()
     .withColumnRenamed('count', 'so_lan_xuat_hien')
     .groupBy('so_lan_xuat_hien').agg(F.count('*').alias('so_bai'))
     .orderBy('so_lan_xuat_hien').show(truncate=False))


Bronze giữ mọi lần fetch là **đúng thiết kế** — mỗi lần fetch là một lần quan sát
riêng. Nhưng con số này định lượng việc Silver phải làm: **dedup theo `link`**.


### 3.2 Có suy ra được quốc gia không?

RSS không có trường địa điểm nào. Cách duy nhất là dò từ khoá trong tiêu đề — đo
xem tỉ lệ dò được là bao nhiêu.


In [ ]:
print('các cột hiện có:',
      [c for c in news.columns if not c.startswith(('_', 'ingestion'))])

KEYWORDS = {
    'Vietnam': r'viet ?nam', 'Thailand': r'thailand|thai\b',
    'Indonesia': r'indonesia', 'Philippines': r'philippine',
    'Malaysia': r'malaysia', 'Singapore': r'singapore',
    'Myanmar': r'myanmar|burma', 'Cambodia': r'cambodia',
    'Laos': r'laos|lao pdr', 'Brunei': r'brunei', 'Timor-Leste': r'timor',
}

haystack = F.lower(F.concat_ws(' ', F.col('title'), F.col('description')))
labelled = news.select('title', 'link', haystack.alias('text')).distinct()

matched = F.lit(False)
for country, pattern in KEYWORDS.items():
    matched = matched | F.col('text').rlike(pattern)
labelled = labelled.withColumn('co_quoc_gia', matched)

total = labelled.count()
hit = labelled.filter('co_quoc_gia').count()
print(f'\nDò được quốc gia: {hit}/{total} bài  ({hit / total:.0%})')
print(f'Không dò được   : {total - hit}/{total} bài  ({1 - hit / total:.0%})')

print('\n--- ví dụ bài KHÔNG dò được quốc gia ---')
labelled.filter(~F.col('co_quoc_gia')).select('title').show(8, truncate=90)


**Kết luận Phần 3.** Tỉ lệ dò được chính là **giới hạn trên** độ chính xác của
nguồn tin tức khi gán về quốc gia. Con số này nên đưa thẳng vào báo cáo — nó cho
thấy nhóm đã *đo* giới hạn của nguồn chứ không giả định nguồn hoàn hảo.


---
## Phần 4 — WHO GHO

`who` đã lọc theo 9/11 mã ISO3 SEA **ở tầng request** (OData `$filter`) — khác với
OpenDengue phải tải hết rồi mới lọc, WHO GHO hỗ trợ lọc server-side thật nên không
cần tải phần dữ liệu ngoài phạm vi về máy.

### 4.1 Độ phủ và độ mới


In [ ]:
print('--- Nước nào có / không có dữ liệu? ---')
co_du_lieu = {row['ISO3'] for row in who.select('ISO3').distinct().collect()}
for iso in SEA_ISO3:
    trang_thai = 'CÓ' if iso in co_du_lieu else 'KHÔNG CÓ'
    print(f'  {iso}: {trang_thai}')

print()
print('--- Ngày mới nhất theo nước ---')
(who.groupBy('ISO3')
    .agg(F.max('START_DATE').alias('moi_nhat'), F.count('*').alias('so_dong'))
    .orderBy('ISO3').show(15, truncate=False))


**Kết luận 4.1 — thiếu 2/11 nước, nhưng mới hơn OpenDengue rất nhiều.**

Philippines (`PHL`) và Brunei (`BRN`) không có dữ liệu — đã xác nhận đây là tình
trạng thật của nguồn (không phải lỗi `$filter`, thử lọc riêng theo tên nước cũng ra
0 dòng). 9 nước còn lại có dữ liệu tới gần thời điểm hiện tại — so với OpenDengue
trễ tới hơn một năm, đây là chênh lệch lớn.

> ⚠️ Có thể thấy vài dòng với `START_DATE` nằm **trong tương lai** so với hôm nay —
> đây là lỗi từ phía nguồn WHO, không phải lỗi ingest. Bronze giữ nguyên (đúng
> nguyên tắc); Silver cần lọc hoặc gắn cờ trước khi dùng.


### 4.2 Cột nào thực sự dùng được?

Bảng có 15 trường ngoài lineage, nhưng không phải trường nào cũng có dữ liệu.


In [ ]:
total = who.count()
cac_cot_phu = ['CONFIRMED_CASES', 'SEVERE_CASES', 'DEATHS',
               'SERO_1', 'SERO_2', 'SERO_3', 'SERO_4', 'POPULATION', 'CASES']
print(f"{'cột':<18}{'% null':>10}")
for col in cac_cot_phu:
    pct = 100 * who.filter(F.col(col).isNull()).count() / total
    print(f'{col:<18}{pct:>9.1f}%')

print()
print('--- DATE_TYPE: độ phân giải thời gian có đồng nhất không? ---')
who.groupBy('DATE_TYPE').count().orderBy(F.desc('count')).show()


**Kết luận 4.2 — chỉ `CASES` dùng được ngay, các cột phụ gần như trống.**

`POPULATION` trống 100%, `SEVERE_CASES` trống hơn 95%, `SERO_1..4` trống hơn 90%
mỗi cột — các trường này tồn tại trong schema nhưng chưa có giá trị thật cho khu
vực SEA. `CASES` là cột duy nhất luôn có giá trị, nên đây là chỉ số chính nên dùng.

Giống hệt vấn đề `T_res` của OpenDengue: `DATE_TYPE` có nhiều loại
(`month`/`isoweek`/`epiweek`) trộn trong cùng bảng — cần chọn một chuẩn hoặc giữ
cột này lại cho Silver quyết định, theo đúng logic đã bàn ở mục 2.2.


---
## Phần 5 — Ghép ba nguồn: vấn đề thật sự

Đây là phát hiện quan trọng nhất của cả notebook.


In [ ]:
print('ĐƠN VỊ ĐỊA LÝ CỦA TỪNG NGUỒN')
print('=' * 68)

print('\n[opendengue] quốc gia + tỉnh/huyện:')
od.select('adm_0_name', 'adm_1_name', 'S_res').distinct().show(3, truncate=False)

print('[news_rss] không có trường địa điểm nào:')
print('  ', [c for c in news.columns if not c.startswith(('_', 'ingestion'))])

print('\n[who_gho] quốc gia (mã ISO3):')
who.select('COUNTRY', 'ISO3').distinct().show(3, truncate=False)


| Nguồn | Đơn vị địa lý | Ghép được không? |
|---|---|---|
| OpenDengue | Quốc gia **hoặc** tỉnh/huyện (`VIET NAM`, `AN GIANG`) | 2 cấp trong cùng bảng |
| WHO GHO | Quốc gia (`ISO3`) | khớp với OpenDengue cấp Admin0 |
| Google News RSS | **Không có** | phải dò từ tiêu đề, chỉ 22% dò được |

**Ba nguồn không cùng một cấp địa lý và không có khoá chung duy nhất.** OpenDengue
và WHO GHO khớp nhau ở cấp quốc gia (cùng dùng được để đối chiếu chéo). Google News
RSS không có trường địa điểm chuẩn nào — phải suy luận, và chỉ đạt ~22% độ chính xác.

### Điều này quyết định thiết kế Silver

Không thể join ba nguồn thành một bảng phẳng duy nhất ngay lập tức. Hai hướng khả dĩ:

1. **Hạ tất cả về cấp quốc gia.** Mất độ chi tiết cấp tỉnh của OpenDengue, nhưng ba
   nguồn xếp chung được một bảng. Đơn giản, làm kịp MVP.
2. **Giữ nhiều cấp trong một bảng**, có cột `admin_level` để phân biệt. Giữ được dữ
   liệu chi tiết nhưng mọi truy vấn sau đều phải lọc theo cấp.

Gợi ý cho MVP: **hướng 1** cho bản đầu tiên, ghi rõ trong báo cáo là đã cân nhắc
hướng 2 — và cấp tỉnh của OpenDengue (mục 2.1–2.2) là ứng viên tự nhiên để mở rộng
sang hướng 2 sau này, vì dữ liệu đã có sẵn trong Bronze.


---
## Phần 6 — Tóm tắt phát hiện

| # | Phát hiện | Ảnh hưởng |
|---|---|---|
| 1 | OpenDengue (`Spatial_extract`, đã lọc SEA) có cấp tỉnh thật — 11/11 nước, hàng trăm tỉnh phân biệt | Đủ điều kiện làm "khoanh vùng nguy cơ" cấp tỉnh, khác bản `National_extract` cũ |
| 2 | `T_res` **CÓ chồng lấn** ở một phần tổ hợp (nước, tỉnh, huyện, năm) khi tính cả cấp tỉnh | `SUM` không lọc `T_res` trước sẽ đếm trùng cục bộ — khác kết luận cũ (chỉ đúng ở cấp quốc gia) |
| 3 | Cấp tỉnh của Việt Nam chủ yếu ở độ phân giải Month, khác cấp quốc gia (có Week từ 2016) | Không giả định "cấp tỉnh mịn như cấp quốc gia" — phải kiểm tra riêng từng cấp |
| 4 | Nguồn ghi `VIET NAM` có dấu cách, bug cũ lọc `VIETNAM` — **đã sửa** ở fetch + `configs/sources.yaml` | Việt Nam không còn bị loại âm thầm |
| 5 | News RSS trùng ~49% giữa các lần fetch, không có trường địa điểm (chỉ 22% dò được từ tiêu đề) | Silver phải dedup theo `link`; tỉ lệ dò quốc gia là giới hạn trên của độ chính xác |
| 6 | WHO GHO thiếu Philippines, Brunei; các cột phụ (`SERO_*`, `POPULATION`, `SEVERE_CASES`) gần như trống | Chỉ `CASES` dùng được ngay; `DATE_TYPE` cũng trộn nhiều loại như `T_res` |
| 7 | Ba nguồn ở nhiều cấp địa lý khác nhau, không có khoá chung duy nhất | Quyết định lớn nhất của thiết kế Silver |

### Việc tiếp theo

- Quyết định cấp địa lý đích của Silver (gợi ý: quốc gia cho MVP, mở rộng tỉnh sau)
- Quyết định grain thời gian đích cho cả OpenDengue lẫn WHO GHO
- Viết `tests/test_integration_news_rss.py`
  (hiện chỉ `opendengue` và `who_gho` có integration test đầy đủ)
- Chụp lại các biểu đồ và bảng trên để đưa vào báo cáo


---
## Phụ lục — dùng dữ liệu có sẵn thay vì tải mới

Nếu đã chạy pipeline trên máy và muốn EDA đúng bộ dữ liệu đó, nén `data/bronze/`
rồi upload:

```python
from google.colab import files
files.upload()                       # chọn bronze.zip
!unzip -q bronze.zip -d /content/data/
```

Rồi đọc lại và bỏ qua Phần 1:

```python
od   = spark.read.format('delta').load('/content/data/bronze/opendengue')
news = spark.read.format('delta').load('/content/data/bronze/news_rss')
who  = spark.read.format('delta').load('/content/data/bronze/who_gho')
```
